## Notes
- Player names were standardised using automated text normalisation to minimise inconsistencies between data sources (FBref and Trnasfermarkt). The resulting merge successfully matched approximately 94% of FBref player records with Transfermarkt.


## Before Merge
- FBref players: 574
- Transfermarkt players: 802
- Merged players: 525

## After Merge
- Successfully merged: 550
- Failed to merge: 12
- Some of the player names were represented by their nicknames. Other players were represented by their first and last names and not their full names.

In [1]:
import pandas as pd
import numpy as np
import unicodedata

from importlib import reload
import transfermarkt_utils

reload(transfermarkt_utils)

from transfermarkt_utils import *

In [2]:
fbref_data = pd.read_csv("/Users/kachi/Documents/Thesis/data/raw/fbref_epl_data.csv")
transfermkt = pd.read_csv("/Users/kachi/Documents/Thesis/data/processed/transfermarkt_epl_data.csv")
#laliga_tm = pd.read_csv("/Users/kachi/Documents/Thesis/data/raw/la_liga_2425.csv")

In [3]:
fbref_data.head()

,league,season,team,player,nation,position,age,born,matches_played,starts,...,wins,draws,losses,clean_sheets,clean_sheet_percentage,penalties_faced,penalties_saved,penalties_missed,penalties_conceded,penalty_save_percentage
0,ENG-Premier League,2122.0,Arsenal,Aaron Ramsdale,ENG,GK,23.0,1998.0,34,34,...,21.0,3.0,10.0,12.0,35.3,6.0,5.0,0.0,1.0,0.0
1,ENG-Premier League,2122.0,Arsenal,Ainsley Maitland-Niles,ENG,MF,23.0,1997.0,8,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENG-Premier League,2122.0,Arsenal,Albert Sambi Lokonga,BEL,MF,21.0,1999.0,19,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENG-Premier League,2122.0,Arsenal,Alexandre Lacazette,FRA,"FW,MF",30.0,1991.0,30,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENG-Premier League,2122.0,Arsenal,Ben White,ENG,DF,23.0,1997.0,32,32,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
fbref_data.shape

(11517, 68)

In [5]:
fbref_data.columns

Index(['league', 'season', 'team', 'player', 'nation', 'position', 'age',
       'born', 'matches_played', 'starts', 'minutes', 'nineties', 'goals',
       'assists', 'goal_contributions', 'non_penalty_goals', 'penalty_goals',
       'penalty_attempts', 'yellow_cards', 'red_cards', 'goals_per90',
       'assists_per90', 'ga_per90', 'npg_per90', 'npga_per90', 'shots',
       'shots_on_target', 'shots_on_target_pct', 'shots_per90',
       'shots_on_target_per90', 'goals_per_shot', 'goals_per_shot_on_target',
       'minutes_per_match', 'minutes_pct', 'starts_per_match',
       'complete_matches', 'substitute_appearances', 'minutes_per_sub',
       'unused_substitute', 'points_per_match', 'team_goals',
       'team_goals_against', 'goal_difference', 'goal_difference_per90',
       'on_off', 'second_yellows', 'fouls_committed', 'fouls_drawn',
       'offsides', 'crosses', 'interceptions', 'tackles_won', 'own_goals',
       'goals_against', 'goals_against_per90', 'shots_on_target_against',


In [6]:
fbref_data['season'].unique()

array([2122., 2223., 2324., 2425.])

In [7]:
fbref_data["league"].isna().sum()

np.int64(0)

In [8]:
fbref_data = fbref_data.rename(
    columns={
        "player": "player_name",
        "team": "club"
    }
)

In [9]:
sorted(fbref_data["club"].unique())

['Ajaccio',
 'Alavés',
 'Almería',
 'Angers',
 'Arminia',
 'Arsenal',
 'Aston Villa',
 'Atalanta',
 'Athletic Club',
 'Atlético Madrid',
 'Augsburg',
 'Auxerre',
 'Barcelona',
 'Bayern Munich',
 'Bochum',
 'Bologna',
 'Bordeaux',
 'Bournemouth',
 'Brentford',
 'Brest',
 'Brighton',
 'Burnley',
 'Cagliari',
 'Celta Vigo',
 'Chelsea',
 'Clermont Foot',
 'Como',
 'Cremonese',
 'Crystal Palace',
 'Cádiz',
 'Darmstadt 98',
 'Dortmund',
 'Elche',
 'Empoli',
 'Espanyol',
 'Everton',
 'Fiorentina',
 'Frankfurt',
 'Freiburg',
 'Frosinone',
 'Fulham',
 'Genoa',
 'Getafe',
 'Girona',
 'Gladbach',
 'Granada',
 'Greuther Fürth',
 'Heidenheim',
 'Hellas Verona',
 'Hertha BSC',
 'Hoffenheim',
 'Holstein Kiel',
 'Inter',
 'Ipswich Town',
 'Juventus',
 'Köln',
 'Las Palmas',
 'Lazio',
 'Le Havre',
 'Lecce',
 'Leeds United',
 'Leganés',
 'Leicester City',
 'Lens',
 'Levante',
 'Leverkusen',
 'Lille',
 'Liverpool',
 'Lorient',
 'Luton Town',
 'Lyon',
 'Mainz 05',
 'Mallorca',
 'Manchester City',
 'Manche

In [10]:
transfermkt.head()

,player_name,position,age,market_value_eur,club,season
0,Emiliano Martínez,Goalkeeper,29.0,28000000.0,Aston Villa,2122
1,Robin Olsen,Goalkeeper,32.0,2000000.0,Aston Villa,2122
2,Jed Steer,Goalkeeper,29.0,600000.0,Aston Villa,2122
3,Viljami Sinisalo,Goalkeeper,20.0,200000.0,Aston Villa,2122
4,Filip Marschall,Goalkeeper,19.0,NaN,Aston Villa,2122


In [11]:
sorted(transfermkt["club"].unique())

['Afc Bournemouth',
 'Arsenal Fc',
 'Aston Villa',
 'Brentford Fc',
 'Brighton And Hove Albion',
 'Burnley Fc',
 'Chelsea Fc',
 'Crystal Palace',
 'Everton Fc',
 'Fulham Fc',
 'Ipswich Town',
 'Leeds United',
 'Leicester City',
 'Liverpool Fc',
 'Luton Town',
 'Manchester City',
 'Manchester United',
 'Newcastle United',
 'Norwich City',
 'Nottingham Forest',
 'Sheffield United',
 'Southampton Fc',
 'Tottenham Hotspur',
 'Watford Fc',
 'West Ham United',
 'Wolverhampton Wanderers']

- Change the club names of Transfermarket to FBRef

In [12]:
# Premier League

club_mapping = {
    "Afc Bournemouth": "Bournemouth",
    "Arsenal Fc": "Arsenal",
    "Brentford Fc": "Brentford",
    "Brighton And Hove Albion": "Brighton",
    "Burnley Fc": "Burnley",
    "Chelsea Fc": "Chelsea",
    "Everton Fc": "Everton",
    "Fulham Fc": "Fulham",
    "Liverpool Fc": "Liverpool",
    "Manchester United": "Manchester Utd",
    "Newcastle United": "Newcastle",
    "Nottingham Forest": "Nottingham",
    "Southampton Fc": "Southampton",
    "Tottenham Hotspur": "Tottenham",
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves",
    "Watford Fc": "Watford"
}

transfermkt["club"] = transfermkt["club"].replace(club_mapping)

#prem_tm['club'] = prem_tm['club'].replace(prem_club_mapping)

# La Liga

In [14]:
prem_fbref = fbref_data[
    fbref_data["league"].str.contains("Premier League")
].copy()

In [15]:
set(prem_fbref["club"]) - set(transfermkt["club"])

set()

In [16]:
sorted(transfermkt["club"].unique())

['Arsenal',
 'Aston Villa',
 'Bournemouth',
 'Brentford',
 'Brighton',
 'Burnley',
 'Chelsea',
 'Crystal Palace',
 'Everton',
 'Fulham',
 'Ipswich Town',
 'Leeds United',
 'Leicester City',
 'Liverpool',
 'Luton Town',
 'Manchester City',
 'Manchester Utd',
 'Newcastle',
 'Norwich City',
 'Nottingham',
 'Sheffield United',
 'Southampton',
 'Tottenham',
 'Watford',
 'West Ham',
 'Wolves']

In [17]:
fbref_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11517 entries, 0 to 11516
Data columns (total 68 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   league                    11517 non-null  object 
 1   season                    11517 non-null  float64
 2   club                      11517 non-null  object 
 3   player_name               11517 non-null  object 
 4   nation                    11513 non-null  object 
 5   position                  11517 non-null  object 
 6   age                       11516 non-null  float64
 7   born                      11516 non-null  float64
 8   matches_played            11517 non-null  int64  
 9   starts                    11517 non-null  int64  
 10  minutes                   11517 non-null  int64  
 11  nineties                  11517 non-null  float64
 12  goals                     11517 non-null  int64  
 13  assists                   11517 non-null  int64  
 14  goal_c

In [18]:
set(fbref_data["club"]) - set(transfermkt["club"])

{'Ajaccio',
 'Alavés',
 'Almería',
 'Angers',
 'Arminia',
 'Atalanta',
 'Athletic Club',
 'Atlético Madrid',
 'Augsburg',
 'Auxerre',
 'Barcelona',
 'Bayern Munich',
 'Bochum',
 'Bologna',
 'Bordeaux',
 'Brest',
 'Cagliari',
 'Celta Vigo',
 'Clermont Foot',
 'Como',
 'Cremonese',
 'Cádiz',
 'Darmstadt 98',
 'Dortmund',
 'Elche',
 'Empoli',
 'Espanyol',
 'Fiorentina',
 'Frankfurt',
 'Freiburg',
 'Frosinone',
 'Genoa',
 'Getafe',
 'Girona',
 'Gladbach',
 'Granada',
 'Greuther Fürth',
 'Heidenheim',
 'Hellas Verona',
 'Hertha BSC',
 'Hoffenheim',
 'Holstein Kiel',
 'Inter',
 'Juventus',
 'Köln',
 'Las Palmas',
 'Lazio',
 'Le Havre',
 'Lecce',
 'Leganés',
 'Lens',
 'Levante',
 'Leverkusen',
 'Lille',
 'Lorient',
 'Lyon',
 'Mainz 05',
 'Mallorca',
 'Marseille',
 'Metz',
 'Milan',
 'Monaco',
 'Montpellier',
 'Monza',
 'Nantes',
 'Napoli',
 'Nice',
 'Osasuna',
 'PSG',
 'Parma',
 'RB Leipzig',
 'Rayo Vallecano',
 'Real Betis',
 'Real Madrid',
 'Real Sociedad',
 'Reims',
 'Rennes',
 'Roma',
 'S

In [19]:
sorted(prem_fbref["club"].unique())

['Arsenal',
 'Aston Villa',
 'Bournemouth',
 'Brentford',
 'Brighton',
 'Burnley',
 'Chelsea',
 'Crystal Palace',
 'Everton',
 'Fulham',
 'Ipswich Town',
 'Leeds United',
 'Leicester City',
 'Liverpool',
 'Luton Town',
 'Manchester City',
 'Manchester Utd',
 'Newcastle',
 'Norwich City',
 'Nottingham',
 'Sheffield United',
 'Southampton',
 'Tottenham',
 'Watford',
 'West Ham',
 'Wolves']

In [20]:
fbref_data.columns

Index(['league', 'season', 'club', 'player_name', 'nation', 'position', 'age',
       'born', 'matches_played', 'starts', 'minutes', 'nineties', 'goals',
       'assists', 'goal_contributions', 'non_penalty_goals', 'penalty_goals',
       'penalty_attempts', 'yellow_cards', 'red_cards', 'goals_per90',
       'assists_per90', 'ga_per90', 'npg_per90', 'npga_per90', 'shots',
       'shots_on_target', 'shots_on_target_pct', 'shots_per90',
       'shots_on_target_per90', 'goals_per_shot', 'goals_per_shot_on_target',
       'minutes_per_match', 'minutes_pct', 'starts_per_match',
       'complete_matches', 'substitute_appearances', 'minutes_per_sub',
       'unused_substitute', 'points_per_match', 'team_goals',
       'team_goals_against', 'goal_difference', 'goal_difference_per90',
       'on_off', 'second_yellows', 'fouls_committed', 'fouls_drawn',
       'offsides', 'crosses', 'interceptions', 'tackles_won', 'own_goals',
       'goals_against', 'goals_against_per90', 'shots_on_target_again

In [21]:
fbref_data["league"].unique()

array(['ENG-Premier League', 'ESP-La Liga', 'FRA-Ligue 1', 'ITA-Serie A',
       'GER-Bundesliga'], dtype=object)

In [22]:
prem_fbref = fbref_data[
    fbref_data["league"] == "ENG-Premier League"
].copy()

In [23]:
prem_fbref.shape

(2269, 68)

In [24]:
sorted(prem_fbref["club"].unique())

['Arsenal',
 'Aston Villa',
 'Bournemouth',
 'Brentford',
 'Brighton',
 'Burnley',
 'Chelsea',
 'Crystal Palace',
 'Everton',
 'Fulham',
 'Ipswich Town',
 'Leeds United',
 'Leicester City',
 'Liverpool',
 'Luton Town',
 'Manchester City',
 'Manchester Utd',
 'Newcastle',
 'Norwich City',
 'Nottingham',
 'Sheffield United',
 'Southampton',
 'Tottenham',
 'Watford',
 'West Ham',
 'Wolves']

In [25]:
prem_fbref["club"] = (
    prem_fbref["club"]
    .replace(club_mapping)
)

In [26]:
# Clean player names in both data sources

#fbref_data["player_name"] = fbref_data["player_name"].apply(clean_player_name)
#transfermkt["player_name"] = transfermkt["player_name"].apply(clean_player_name)

prem_fbref["player_name"] = (
    prem_fbref["player_name"]
    .apply(clean_player_name)
)

transfermkt["player_name"] = (
    transfermkt["player_name"]
    .apply(clean_player_name)
)

# Merge both data sources

#merged = fbref_data.merge(
    #transfermkt,
   # on=["player_name", "club", "season"],
  #  how="left",
 #   suffixes=("_fbref", "_tm")
#)


merged = prem_fbref.merge(
    transfermkt,
    on=["player_name", "club", "season"],
    how="left",
    suffixes=("_fbref", "_tm")
)

In [27]:
merged.shape

(2269, 71)

In [28]:
merged["market_value_eur"].notna().sum()

np.int64(2116)

In [29]:
match_rate = (
    merged["market_value_eur"].notna().mean() * 100
)

print(f"Match rate: {match_rate:.2f}%")

Match rate: 93.26%


In [30]:
unmatched = merged[
    merged["market_value_eur"].isna()
]


unmatched[
    ["player_name", "club", "season"]
].head(20)

,player_name,club,season
12,gabriel magalhaes,Arsenal,2122.0
36,emi buendia,Aston Villa,2122.0
40,jaden philogene bidace,Aston Villa,2122.0
65,finley stevens,Brentford,2122.0
76,mathias jrgensen,Brentford,2122.0
77,nathan young coombes,Brentford,2122.0
86,alvaro fernandez,Brentford,2122.0
127,johann berg gumundsson,Burnley,2122.0
142,emerson palmieri,Chelsea,2122.0
202,isaac price,Everton,2122.0


In [31]:
transfermkt[
    transfermkt["club"] == "Arsenal"
].query("player_name.str.contains('gab', case=False)", engine="python")

,player_name,position,age,market_value_eur,club,season
768,gabriel,Centre-Back,24.0,38000000.0,Arsenal,2122
795,gabriel martinelli,Left Winger,21.0,40000000.0,Arsenal,2122
857,gabriel,Centre-Back,25.0,55000000.0,Arsenal,2223
882,gabriel martinelli,Left Winger,22.0,80000000.0,Arsenal,2223
890,gabriel jesus,Centre-Forward,26.0,75000000.0,Arsenal,2223
1693,gabriel,Centre-Back,26.0,70000000.0,Arsenal,2324
1718,gabriel martinelli,Left Winger,23.0,70000000.0,Arsenal,2324
1725,gabriel jesus,Centre-Forward,27.0,65000000.0,Arsenal,2324
2546,gabriel,Centre-Back,27.0,75000000.0,Arsenal,2425
2571,gabriel martinelli,Left Winger,24.0,55000000.0,Arsenal,2425


In [32]:
transfermkt[
    transfermkt["club"] == "Manchester Utd"
].query("player_name.str.contains('garn', case=False)", engine="python")

,player_name,position,age,market_value_eur,club,season
460,james garner,Defensive Midfield,21.0,7000000.0,Manchester Utd,2122
477,alejandro garnacho,Left Winger,17.0,NaN,Manchester Utd,2122
1367,james garner,Defensive Midfield,22.0,14000000.0,Manchester Utd,2223
1379,alejandro garnacho,Left Winger,18.0,25000000.0,Manchester Utd,2223
2229,alejandro garnacho,Left Winger,19.0,45000000.0,Manchester Utd,2324
3065,alejandro garnacho,Left Winger,20.0,45000000.0,Manchester Utd,2425


In [33]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2269 entries, 0 to 2268
Data columns (total 71 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   league                    2269 non-null   object 
 1   season                    2269 non-null   float64
 2   club                      2269 non-null   object 
 3   player_name               2269 non-null   object 
 4   nation                    2267 non-null   object 
 5   position_fbref            2269 non-null   object 
 6   age_fbref                 2268 non-null   float64
 7   born                      2268 non-null   float64
 8   matches_played            2269 non-null   int64  
 9   starts                    2269 non-null   int64  
 10  minutes                   2269 non-null   int64  
 11  nineties                  2269 non-null   float64
 12  goals                     2269 non-null   int64  
 13  assists                   2269 non-null   int64  
 14  goal_con

In [34]:
merged.head()

,league,season,club,player_name,nation,position_fbref,age_fbref,born,matches_played,starts,...,clean_sheets,clean_sheet_percentage,penalties_faced,penalties_saved,penalties_missed,penalties_conceded,penalty_save_percentage,position_tm,age_tm,market_value_eur
0,ENG-Premier League,2122.0,Arsenal,aaron ramsdale,ENG,GK,23.0,1998.0,34,34,...,12.0,35.3,6.0,5.0,0.0,1.0,0.0,Goalkeeper,24.0,28000000.0
1,ENG-Premier League,2122.0,Arsenal,ainsley maitland niles,ENG,MF,23.0,1997.0,8,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Right-Back,24.0,10000000.0
2,ENG-Premier League,2122.0,Arsenal,albert sambi lokonga,BEL,MF,21.0,1999.0,19,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Central Midfield,22.0,14000000.0
3,ENG-Premier League,2122.0,Arsenal,alexandre lacazette,FRA,"FW,MF",30.0,1991.0,30,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Centre-Forward,31.0,15000000.0
4,ENG-Premier League,2122.0,Arsenal,ben white,ENG,DF,23.0,1997.0,32,32,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Right-Back,24.0,40000000.0


In [35]:
merged["market_value_eur"].notna().sum()

np.int64(2116)

In [36]:
merged["market_value_eur"].isna().sum()

np.int64(153)

In [37]:
merged.duplicated(
    subset=["player_name", "club", "season"]
).sum()

np.int64(0)

In [38]:
merged.groupby(
    ["player_name", "club", "season"]
).size().sort_values(ascending=False).head(20)

player_name       club            season
aaron connolly    Brighton        2122.0    1
matt doherty      Tottenham       2223.0    1
mathias normann   Norwich City    2122.0    1
mathis amougou    Chelsea         2425.0    1
mathys tel        Tottenham       2425.0    1
matias vina       Bournemouth     2223.0    1
mats wieffer      Brighton        2425.0    1
matt doherty      Tottenham       2122.0    1
                  Wolves          2324.0    1
mathias jrgensen  Brentford       2223.0    1
matt doherty      Wolves          2425.0    1
matt o riley      Brighton        2425.0    1
matt ritchie      Newcastle       2122.0    1
                                  2223.0    1
                                  2324.0    1
matt targett      Aston Villa     2122.0    1
mathias jrgensen  Brentford       2324.0    1
                                  2122.0    1
mateus fernandes  Southampton     2425.0    1
matheus franca    Crystal Palace  2425.0    1
dtype: int64

In [39]:
merged[
    merged["market_value_eur"].notna()
][[
    "player_name",
    "club",
    "season",
    "market_value_eur"
]].sample(10, random_state=42)

,player_name,club,season,market_value_eur
1737,lucas digne,Aston Villa,2425.0,10000000.0
1573,ryan yates,Nottingham,2324.0,12000000.0
271,timothy castagne,Leicester City,2122.0,28000000.0
457,oliver skipp,Tottenham,2122.0,18000000.0
1611,william osula,Sheffield United,2324.0,3000000.0
465,adam masina,Watford,2122.0,3000000.0
1142,calum chambers,Aston Villa,2324.0,4000000.0
767,vitaliy mykolenko,Everton,2223.0,25000000.0
933,alexander isak,Newcastle,2223.0,70000000.0
1272,jacob bruun larsen,Burnley,2324.0,8000000.0


In [40]:
merged.to_csv(
    "/Users/kachi/Documents/Thesis/data/processed/fbref_transfermarkt_merged.csv",
    index=False
)

In [126]:
set(fbref_2425["club"]) == set(prem_tm["club"])

False

In [127]:
prem_tm.head()

,player_name,position,Age,market_value_eur,club,season
0,Kepa Arrizabalaga,Goalkeeper,30.0,10000000.0,Bournemouth,2425
1,Mark Travers,Goalkeeper,26.0,4000000.0,Bournemouth,2425
2,Neto,Goalkeeper,35.0,1500000.0,Bournemouth,2425
3,Callan McKenna,Goalkeeper,18.0,600000.0,Bournemouth,2425
4,Will Dennis,Goalkeeper,24.0,350000.0,Bournemouth,2425


In [128]:
fbref_2425.head()

,league,season,club,player_name,nation,position,age,born,matches_played,starts,...,non_penalty_goals,penalty_goals,penalty_attempts,yellow_cards,red_cards,goals_per90,assists_per90,ga_per90,npg_per90,npga_per90
1693,ENG-Premier League,2425,Arsenal,Ben White,ENG,DF,26.0,1997.0,17,13,...,0,0,0,2,0,0.00,0.15,0.15,0.00,0.15
1694,ENG-Premier League,2425,Arsenal,Bukayo Saka,ENG,"FW,MF",22.0,2001.0,25,20,...,5,1,1,3,0,0.31,0.52,0.83,0.26,0.78
1695,ENG-Premier League,2425,Arsenal,David Raya,ESP,GK,28.0,1995.0,38,38,...,0,0,0,3,0,0.00,0.00,0.00,0.00,0.00
1696,ENG-Premier League,2425,Arsenal,Declan Rice,ENG,MF,25.0,1999.0,35,33,...,4,0,0,7,1,0.13,0.22,0.35,0.13,0.35
1697,ENG-Premier League,2425,Arsenal,Ethan Nwaneri,ENG,"FW,MF",17.0,2007.0,26,11,...,4,0,0,1,0,0.40,0.20,0.60,0.40,0.60


In [129]:
fbref_2425["league"].unique()

array(['ENG-Premier League', 'ESP-La Liga', 'FRA-Ligue 1', 'ITA-Serie A',
       'GER-Bundesliga', nan], dtype=object)

In [130]:
fbref_2425[
    fbref_2425['league'].isna()
].head()

,league,season,club,player_name,nation,position,age,born,matches_played,starts,...,non_penalty_goals,penalty_goals,penalty_attempts,yellow_cards,red_cards,goals_per90,assists_per90,ga_per90,npg_per90,npga_per90
11136,NaN,2425,Frankfurt,Ansgar Knauff,GER,MF,22.0,2002.0,30,18,...,4,0,0,3,0,0.23,0.28,0.51,0.23,0.51
11137,NaN,2425,Frankfurt,Arthur Theate,BEL,DF,24.0,2000.0,31,31,...,0,0,0,6,1,0.00,0.00,0.00,0.00,0.00
11138,NaN,2425,Frankfurt,Aurèle Amenda,SUI,DF,21.0,2003.0,8,1,...,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00
11139,NaN,2425,Frankfurt,Can Uzun,TUR,MF,18.0,2005.0,20,6,...,4,0,0,1,0,0.53,0.13,0.67,0.53,0.67
11140,NaN,2425,Frankfurt,Ellyes Skhiri,TUN,MF,29.0,1995.0,30,26,...,1,0,0,6,0,0.04,0.08,0.12,0.04,0.12


In [95]:
fbref_2425.loc[
    fbref_2425["league"].isna(),
    "club"
].unique()

array(['Frankfurt'], dtype=object)

In [72]:
fbref_2425.loc[
    fbref_2425["league"].isna(),
    "club"
].nunique()

1

In [73]:
fbref_2425.loc[
    fbref_2425["league"].isna()
].shape

(26, 25)

In [40]:
fbref_2425.loc[
    fbref_2425["league"].isna(),
    "league"
] = "GER-Bundesliga"

In [41]:
fbref_2425["league"].isna().sum()

np.int64(0)

In [18]:
fbref_2425["club"].unique()

array(['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton',
       'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich Town',
       'Leicester City', 'Liverpool', 'Manchester City', 'Manchester Utd',
       'Newcastle United', 'Nottingham Forest', 'Southampton',
       'Tottenham Hotspur', 'West Ham United', 'Wolves', 'Alavés',
       'Athletic Club', 'Atlético Madrid', 'Barcelona', 'Celta Vigo',
       'Espanyol', 'Getafe', 'Girona', 'Las Palmas', 'Leganés',
       'Mallorca', 'Osasuna', 'Rayo Vallecano', 'Real Betis',
       'Real Madrid', 'Real Sociedad', 'Sevilla', 'Valencia',
       'Valladolid', 'Villarreal', 'Angers', 'Auxerre', 'Brest',
       'Le Havre', 'Lens', 'Lille', 'Lyon', 'Marseille', 'Monaco',
       'Montpellier', 'Nantes', 'Nice', 'Paris Saint-Germain', 'Reims',
       'Rennes', 'Saint-Étienne', 'Strasbourg', 'Toulouse', 'Atalanta',
       'Bologna', 'Cagliari', 'Como', 'Empoli', 'Fiorentina', 'Genoa',
       'Hellas Verona', 'Inter', 'Juventus',

In [19]:
prem_tm["club"].unique()

array(['Bournemouth', 'Arsenal', 'Aston Villa', 'Brentford', 'Brighton',
       'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich Town',
       'Leicester City', 'Liverpool', 'Manchester City', 'Manchester Utd',
       'Newcastle United', 'Nottingham Forest', 'Southampton',
       'Tottenham Hotspur', 'West Ham United', 'Wolves'], dtype=object)

In [17]:
# FBref Data For EPL Only

fbref_prem = fbref_2425[
    fbref_2425["league"] == "ENG-Premier League"
].copy()


# FBref Data For La Liga Only

fbref_laliga = fbref_2425[
    fbref_2425["league"] == "ESP-La Liga"
].copy()


## Check if the set for tm for each league are the same as the set in fbref

In [18]:
set(fbref_2425["club"]) == set(prem_tm["club"])

False

In [20]:
set(fbref_2425["club"]) == set(laliga_tm["club"])

False

## Checking the difference between the datasets

In [21]:
set(fbref_prem["club"]) - set(prem_tm["club"])

set()

In [22]:
set(prem_tm["club"]) - set(fbref_prem["club"])

set()

In [23]:
set(fbref_laliga["club"]) - set(laliga_tm["club"])

{'Alavés',
 'Athletic Club',
 'Atlético Madrid',
 'Barcelona',
 'Celta Vigo',
 'Espanyol',
 'Getafe',
 'Girona',
 'Las Palmas',
 'Leganés',
 'Mallorca',
 'Osasuna',
 'Real Betis',
 'Sevilla',
 'Valencia',
 'Valladolid',
 'Villarreal'}

In [24]:
set(laliga_tm["club"]) - set(fbref_laliga["club"])

{'Athletic Bilbao',
 'Atletico De Madrid',
 'Ca Osasuna',
 'Cd Leganes',
 'Celta De Vigo',
 'Deportivo Alaves',
 'Fc Barcelona',
 'Getafe Cf',
 'Girona Fc',
 'Rcd Espanyol Barcelona',
 'Rcd Mallorca',
 'Real Betis Balompie',
 'Real Valladolid Cf',
 'Sevilla Fc',
 'Ud Las Palmas',
 'Valencia Cf',
 'Villarreal Cf'}

## Manual Maping to make club names to be consistent across data sources

In [29]:
# La liga

laliga_club_mapping = {
    "Deportivo Alaves": "Alavés",
    "Athletic Bilbao": "Athletic Club",
    "Atletico De Madrid": "Atlético Madrid",
    "Fc Barcelona": "Barcelona",
    "Celta De Vigo": "Celta Vigo",
    "Rcd Espanyol Barcelona": "Espanyol",
    "Getafe Cf": "Getafe",
    "Girona Fc": "Girona",
    "Ud Las Palmas": "Las Palmas",
    "Cd Leganes": "Leganés",
    "Rcd Mallorca": "Mallorca",
    "Ca Osasuna": "Osasuna",
    "Real Betis Balompie": "Real Betis",
    "Sevilla Fc": "Sevilla",
    "Valencia Cf": "Valencia",
    "Real Valladolid Cf": "Valladolid",
    "Villarreal Cf": "Villarreal"
}

laliga_tm["club"] = laliga_tm["club"].replace(laliga_club_mapping)

set(fbref_laliga["club"]) - set(laliga_tm["club"])

set()

In [28]:
set(laliga_tm["club"]) - set(fbref_laliga["club"])

set()

## FBref names

This includes all active players and not registered players for the 24-25 season

In [23]:
set(fbref_prem["player_name"]) - set(prem_tm["player_name"])

{'Abdul Fatawu Issahaku',
 'Albert Grønbaek',
 'Ali Al Hamadi',
 'Andy Irving',
 'Arijanet Muric',
 'Armel Bella Kotchap',
 'Ben Brereton',
 'Chidozie Obi-Martin',
 'Danilo Santos',
 'Edmond-Paris Maghoma',
 'Emerson Palmieri',
 'Emi Buendía',
 'Fabio Carvalho',
 'Ferdi Kadioglu',
 'Gabriel Magalhães',
 'Hwang Hee-chan',
 'Idrissa Gana Gueye',
 'Igor',
 'Illia Zabarnyi',
 'Ismaila Sarr',
 'Jaden Philogene Bidace',
 'James Mcatee',
 'Jeremy Doku',
 'Joshua Acheampong',
 'Joshua King',
 'Joško Gvardiol',
 'Julian Araujo',
 'Jáder Durán',
 'Kim Jisoo',
 'Kosta Nedeljković',
 'Kostas Tsimikas',
 'Luis Guilherme',
 'Mateo Kovačić',
 'Mateus Mane',
 'Max Kilman',
 'Mykhailo Mudryk',
 'Nathan Wood-Gordon',
 'Nico O’Reilly',
 'Nicolás González',
 'Son Heung-min',
 'Sávio',
 'Toti Gomes',
 'Valentino Livramento',
 'Victor Bernth Kristiansen',
 'William Smallbone',
 'Yehor Yarmoliuk',
 'Yunus Emre Konak',
 'Łukasz Fabiański'}

## Transfermarkt names

This includes all registered players and not active players for the 24-25 season

In [24]:
set(prem_tm["player_name"]) - set(fbref_prem["player_name"])

{'Aaron Hickey',
 'Aarón Anselmino',
 'Abdul Fatawu',
 'Aidan Borland',
 'Albert Grønbæk',
 'Alex Murphy',
 'Alexéi Rojas',
 'Alfie Devine',
 'Alfie Whiteman',
 'Ali Al-Hamadi',
 'Amara Nallo',
 'Amario Cozier-Duberry',
 'Andrew Irving',
 'Andrew Omobamidele',
 'Andrey Santos',
 'Archie Harris',
 'Arijanet Murić',
 'Armel Bella-Kotchap',
 'Asmir Begovic',
 'Ato Ampah',
 'Bastien Meupiyou',
 'Ben Brereton Díaz',
 'Ben Broggio',
 'Ben Nelson',
 'Ben Perry',
 'Ben Winterbottom',
 'Bendito Mantato',
 'Benjamin Arthur',
 'Benjamin Fredrick',
 'Bradley Burrowes',
 'Bradley Moonan',
 'Brayden Clarke',
 'Callan McKenna',
 'Callum Bates',
 'Callum Olusesi',
 'Cameron Peupion',
 'Carl Rushworth',
 'Carlos Miguel',
 'Carney Chukwuemeka',
 'Cesare Casadei',
 'Charlie Tasker',
 'Chem Campbell',
 'Chido Obi',
 'Chris Mepham',
 'Chris Popov',
 'Cieran Slicker',
 'Coby Ebere',
 'Damola Ajayi',
 'Dan Gore',
 'Daniel Adu-Adjei',
 'Daniel Iversen',
 'Danilo',
 'Dante Cassanova',
 'David Datro Fofana',
 '

## Merging both FBref and Transfermarkt

In [25]:
merged_df = fbref_prem.merge(
    prem_tm,
    on=["player_name", "club", "season"],
    how="inner"
)

In [26]:
print(f"FBref players: {len(fbref_prem)}")
print(f"Transfermarkt players: {len(prem_tm)}")
print(f"Merged players: {len(merged_df)}")

FBref players: 574
Transfermarkt players: 802
Merged players: 525


In [31]:
fbref_prem["player_clean"] = (
    fbref_prem["player_name"]
    .apply(clean_player_name)
)

In [32]:
prem_tm["player_clean"] = (
    prem_tm["player_name"]
    .apply(clean_player_name)
)

In [33]:
merged_df = fbref_prem.merge(
    prem_tm,
    left_on=[
        "player_clean",
        "club",
        "season"
    ],
    right_on=[
        "player_clean",
        "club",
        "season"
    ],
    how="inner"
)

In [34]:
print(len(merged_df))

538


In [35]:
remaining = (
    set(fbref_prem["player_clean"])
    - set(merged_df["player_clean"])
)

len(remaining)

35

In [36]:
sorted(remaining)

['abdul fatawu issahaku',
 'albert grnbaek',
 'andy irving',
 'ben brereton',
 'chidozie obi martin',
 'danilo santos',
 'edmond paris maghoma',
 'emerson palmieri',
 'emi buendia',
 'ferdi kadioglu',
 'gabriel magalhaes',
 'hwang hee chan',
 'idrissa gana gueye',
 'igor',
 'illia zabarnyi',
 'jaden philogene bidace',
 'jader duran',
 'joshua acheampong',
 'joshua king',
 'kim jisoo',
 'kostas tsimikas',
 'max kilman',
 'mykhailo mudryk',
 'nathan wood gordon',
 'nico oreilly',
 'nicolas gonzalez',
 'savio',
 'son heung min',
 'toti gomes',
 'ukasz fabianski',
 'valentino livramento',
 'victor bernth kristiansen',
 'william smallbone',
 'yehor yarmoliuk',
 'yunus emre konak']

## Handling the remaining 35 players that were not merged

- Some of the player names were represented by their nicknames.
- Other players were represented by their first and last names and not their full names.

In [44]:
fbref_prem["player_clean"] = (
    fbref_prem["player_clean"]
    .replace(PLAYER_NAME_MAPPING)
)

## Merging with reusable function for all the 5 leagues (24-25 season)

In [45]:
prem_master = merge_fbref_transfermarkt(
    fbref_prem,
    prem_tm
)

seriea_master = merge_fbref_transfermarkt(
    fbref_seriea,
    transfermarkt_seriea
)

In [46]:
prem_master.shape

(550, 30)

In [ ]:
laliga_master = merge_fbref_transfermarkt(
    fbref_laliga,
    laliga_tm
)

print(len(fbref_laliga))
print(len(laliga_tm))
print(len(laliga_master))